# 06 — ERA5 気候値パックを GCP 側で作る

**このタスクだけは GCP（Colab）で実行する。** 理由は単純で、データが Google Cloud Storage に
置かれているから:

| やり方 | 転送量（1変数あたり） |
|---|---|
| ARO-ERA5 から日本の手元で 30 年分を自前計算 | 約 45 GB の読み込み |
| **Colab（= GCS の隣）で計算し、成果物だけ持ち帰る** | **パック数百 MB のダウンロードのみ** |
| **WeatherBench2 の計算済み気候値を使う（方式 A）** | 再計算ゼロ、サブセットのみ |

このノートブックは Colab でそのまま動く（ローカルの `./.venv` でも点検用の小さな読み込みは可能）。

**出力**: アプリの気候値キャッシュ形式
`~/.cache/aiseed-weather/climatology/{var}_{ref}_{smoothing}.nc`
（`climatology-analysis` スキルのキー規約に一致。ユーザーはこのファイルを置くだけで
アノマリ機能が即使える = 「一度 GCP で計算し、みんなに配る」）

## 方式 A: WeatherBench2 の計算済み ERA5 気候値（推奨・数分）

Google Research が WeatherBench2 で **ERA5 の時別気候値（1990–2019・0.25° フル解像度・
平滑化済み）** を公開している。これをサブセット・変換するだけで済む。

- ストア: `gs://weatherbench2/datasets/era5-hourly-climatology/1990-2019_6h_1440x721.zarr`
- 次元: `(hour: 4, dayofyear: 366, latitude: 721, longitude: 1440, level: 13)`
- **注意: 参照期間は 1990–2019**（WMO 標準の 1991–2020 ではない）。
  図のラベルには必ず「vs 1990-2019」と明記すること。WMO 期間が必要なら方式 B へ。

In [ ]:
# セットアップ（Colab では依存を導入。ローカル ./.venv なら既に揃っている）
import importlib.util, sys
IN_COLAB = "google.colab" in sys.modules
if IN_COLAB:
    %pip -q install xarray zarr fsspec aiohttp requests netcdf4 matplotlib

import fsspec
import numpy as np
import xarray as xr

WB2_CLIM = ("https://storage.googleapis.com/weatherbench2/datasets/"
            "era5-hourly-climatology/1990-2019_6h_1440x721.zarr")
ds = xr.open_zarr(fsspec.get_mapper(WB2_CLIM), consolidated=True)
print(dict(ds.sizes))

In [ ]:
# パック対象（アプリの「最初に要る」層: climatology-analysis スキル参照）
#   msl / 2t は単一層、gh500 は geopotential@500 を重力加速度で割って gpm に、t850 は temperature@850
TARGETS = {
    "msl":  ds["mean_sea_level_pressure"],                       # Pa
    "2t":   ds["2m_temperature"],                                # K
    "gh500": ds["geopotential"].sel(level=500) / 9.80665,        # m2/s2 → gpm
    "t850": ds["temperature"].sel(level=850),                    # K
}
REF = "1990_2019"          # WB2 の参照期間（ラベル用）
SMOOTHING = "wb2"          # WB2 側で平滑化済み
COARSEN = 2                # 1 = 0.25°のまま（大きい） / 2 = 0.5°（推奨・約1/4）
OUT_DIR = "climatology"    # Colab 上の出力先ディレクトリ

In [ ]:
# 日別気候値に落として int16 CF パッキングで保存
#   - hour 次元は平均して「日別」に（アプリの日別アノマリ用）
#   - CF の scale_factor/add_offset パッキングなので xarray で開けば自動的に実数に戻る
import os
os.makedirs(OUT_DIR, exist_ok=True)

def pack(name, da):
    daily = da.mean("hour")
    if COARSEN > 1:
        daily = daily.coarsen(latitude=COARSEN, longitude=COARSEN, boundary="trim").mean()
    daily = daily.compute()
    vmin, vmax = float(daily.min()), float(daily.max())
    scale = (vmax - vmin) / 65000 or 1.0
    offset = (vmax + vmin) / 2
    path = f"{OUT_DIR}/{name}_{REF}_{SMOOTHING}.nc"
    daily.to_dataset(name=name).to_netcdf(path, encoding={name: {
        "dtype": "int16", "scale_factor": scale, "add_offset": offset,
        "_FillValue": -32768, "zlib": True, "complevel": 5, "shuffle": True}})
    print(f"{path}: {os.path.getsize(path)/1e6:.1f} MB (範囲 {vmin:.1f}〜{vmax:.1f})")

for name, da in TARGETS.items():
    pack(name, da)

In [ ]:
# 検証: 東京上空の年間カーブ（値の妥当性チェック）
import matplotlib
import matplotlib.pyplot as plt

chk = xr.open_dataset(f"{OUT_DIR}/msl_{REF}_{SMOOTHING}.nc")
tokyo = chk["msl"].sel(latitude=35.75, longitude=139.75, method="nearest") / 100
fig, ax = plt.subplots(figsize=(8, 3))
ax.plot(tokyo["dayofyear"], tokyo)
ax.set_title("東京 MSL 日別気候値 (vs 1990-2019, WB2)  ※冬高・夏低なら正しい")
ax.set_xlabel("day of year"); ax.set_ylabel("hPa"); ax.grid(alpha=0.3)
plt.tight_layout(); plt.show()
print("1月1日:", float(tokyo.sel(dayofyear=1)), "hPa / 7月1日:", float(tokyo.sel(dayofyear=182)), "hPa")

In [ ]:
# 持ち帰り: Colab なら Drive へコピー（またはブラウザダウンロード）
if IN_COLAB:
    from google.colab import drive
    drive.mount("/content/drive")
    import shutil
    dst = "/content/drive/MyDrive/aiseed-weather-climatology"
    shutil.copytree(OUT_DIR, dst, dirs_exist_ok=True)
    print("Drive に保存:", dst)
else:
    print("ローカル実行: ./climatology/ に出力済み。"
          "~/.cache/aiseed-weather/climatology/ に置けばアプリがそのまま使う")

## 方式 B: WMO 標準の 1991–2020 を厳密に使う場合（ARCO-ERA5 から自前計算）

`climatology-analysis` スキルの既定は **1991–2020（WMO 現行平年値）**。WB2 の 1990–2019 で
実用上ほぼ差はないが、公表物で WMO 準拠を名乗るならこちらで作る。

- ソース: `gs://gcp-public-data-arco-era5/ar/full_37-1h-0p25deg-chunk-1.zarr-v3`
- 手順: 12UTC を日代表として `sel` → `groupby("dayofyear").mean/std` → 31 日移動平均
- **必ず Colab/GCP 上で**（時間チャンク=1 のため読み込みが桁違い。日本へは成果物のみ）
- 長時間ジョブになるため、変数ごとにランタイムを分けるのが安全

```python
ar = xr.open_zarr(fsspec.get_mapper(
    "https://storage.googleapis.com/gcp-public-data-arco-era5/ar/full_37-1h-0p25deg-chunk-1.zarr-v3"),
    consolidated=True)
z500 = ar["geopotential"].sel(level=500).sel(time=ar.time.dt.hour == 12) \
         .sel(time=slice("1991-01-01", "2020-12-31"))
clim = z500.groupby("time.dayofyear").mean("time")           # + .std("time") で σ も
clim = clim.rolling(dayofyear=31, center=True, min_periods=1).mean()   # スキル準拠の平滑
```

## 出典表記（必須）

- ERA5: *Hersbach et al. (2020), Copernicus Climate Change Service (C3S)* —
  「Contains modified Copernicus Climate Change Service information」
- WeatherBench2 climatology: *Rasp et al. (2024), WeatherBench 2*
- 図には参照期間（1990-2019 または 1991-2020）と平滑化の有無を必ず記載する
  （`climatology-analysis` スキルの Labels 規約）。